# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     print(f'The number of tokens the cell called: {response.usage}')
     return response.choices[0].message.content
response=ask_llm('how many days are in a leap year')
print(response)
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

The number of tokens the cell called: CompletionUsage(completion_tokens=54, prompt_tokens=49, total_tokens=103, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.01022388, prompt_time=0.001458977, completion_time=0.165866386, total_time=0.167325363)
There are 366 days in a leap year. This is one more day than in a standard year (365 days), with the extra day being added to the month of February, which has 29 days in a leap year instead of the usual 28 days.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
Ans: The system role gives the model its overall instructions or behavior, while the user role contains the actual question or request. For example, the system could say “You are a helpful assistant”, while the user could ask “How many days are in a leap year?”


*2. What is a token, roughly? Why do API providers bill per token rather than per request?*
A token is a small piece of text, such as part of a word or a whole word. API providers charge per token because longer prompts and responses require more processing than shorter ones.
> **Answer:** [Double-click to edit]

### Part 1.2 — Temperature: the randomness dial

In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a savings product for market traders in Accra."
for i in range(5):
  answer = ask_llm(test_question, temperature=0.0)
  print(f"Answer {i + 1}: {answer}")

for i in range(5):
  answer = ask_llm(test_question, temperature=1.2)
  print(f"Answer {i + 1}: {answer}")

# TODO: Print all 10 answers, grouped by temperature.

The number of tokens the cell called: CompletionUsage(completion_tokens=256, prompt_tokens=56, total_tokens=312, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010811652, prompt_time=0.001929657, completion_time=0.742284829, total_time=0.744214486)
Answer 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Trader

**Student Reasoning — Temperature**
*What did you observe at each temperature?
At temperature 0, the answers were more consistent and predictable. At temperature 1.2, the answers varies.

For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*
For a loan decision-support system, a low temperature such as 0 is more appropriate because the system should give consistent and reliable answers rather than the unpredictable ones.

> **Answer:** [Double-click to edit]

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [6]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1="Summarize this:"
V1_on_L002= ask_llm(SUMMARY_PROMPT_V1+ LETTERS.get("L002"))
print(V1_on_L002)

V1_on_L006 = ask_llm(SUMMARY_PROMPT_V1 + LETTERS.get("L006"))
print(V1_on_L006)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_PROMPT_V2= "You are an assistant to a microfinance loan officer. Give a factual and neutral summary of the loan application. Do not invent any details. Keep the summary between 3 and 4 sentences"
V2_prompt_L002 = f"Summarize this loan application:\n\n{LETTERS.get('L002')}"
V2_prompt_L006 = f"Summarize this loan application:\n\n{LETTERS.get('L006')}"

V2_on_L002 = ask_llm(V2_prompt_L002, SUMMARY_PROMPT_V2,temperature=0)
V2_on_L006 = ask_llm(V2_prompt_L006, SUMMARY_PROMPT_V2, temperature=0)
print(V2_on_L002)
print(V2_on_L006)

print(f'V1 output on L002: {V1_on_L002}')
print(f'V2 output on L002: {V2_on_L002}')

print(f'V1 output on L006: {V1_on_L006}')
print(f'V2 output on L006: {V2_on_L006}')

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

The number of tokens the cell called: CompletionUsage(completion_tokens=71, prompt_tokens=133, total_tokens=204, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.009927594, prompt_time=0.006430076, completion_time=0.206134458, total_time=0.212564534)
Kwame Boateng, a commercial driver in Kumasi, is requesting a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business but expects it to improve after the festive season, and is willing to repay the loan when he can, despite not having collateral.
The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=135, total_tokens=210, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.009877475, prompt_time=0.006214591, completion_time=0.228758593, total_time=0.234973184)
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car wash, a provision shop, and a phone import business

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
Ans: V1 was less specific and sounded a bit more dramatic in some places. For example, it said Kwame was “urgently seeking GHS 25,000,” even though the letter did not really say that. V2 was better because it simply said he “has applied for a loan of GHS 25,000.” This made V2 more factual and closer to what was actually written in the letter.




*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

Ans: No invented details is important because the model should only use information that the applicant actually gave. If it adds something that was never mentioned, the loan officer could make a wrong decision based on false information. The failure mode is called hallucination in LLMs.

> **Answer:** [Double-click to edit]

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [7]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT='''return ONLY a JSON object with EXACTLY these keys:
applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
repayment_months (number or null)
If a field is not stated in the letter, use null. Do not guess.
example:
letter: My name is Wilson. I need about 10000 Ghana cedis to start a business and i expect to make 20000 Ghana cedis in profit every month. I will give my phone as a collateral and i will  pay back in a year.


JSON:{
    "applicant_name": "Wilson",
    "amount_ghs": 10000,
    "purpose": "start a business",
    "monthly_profit_ghs": 20000,
    "has_collateral_or_guarantor": true,
    "repayment_months": 12}'''

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
import pandas as pd
import json

def extract_fields(letter_text):
  prompt=f"{EXTRACT_PROMPT}\nletter:{letter_text}"
  answer=ask_llm(prompt,temperature=0)
  answer = answer.replace("```json", "")
  answer=answer.replace("```", "")
  answer=answer.strip()
  try:
    result=json.loads(answer)
    return result
  except json.JSONDecodeError:
    print(f"Parse error: {answer}")
    return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

result=[]
for i in LETTERS:
  result.append(extract_fields(LETTERS.get(i)))
df=pd.DataFrame(result)
print(df)


The number of tokens the cell called: CompletionUsage(completion_tokens=76, prompt_tokens=362, total_tokens=438, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.01034569, prompt_time=0.018488555, completion_time=0.115569347, total_time=0.134057902)
The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=322, total_tokens=397, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.009617357, prompt_time=0.016274454, completion_time=0.116658048, total_time=0.132932502)
The number of tokens the cell called: CompletionUsage(completion_tokens=80, prompt_tokens=376, total_tokens=456, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.009542731, prompt_time=0.018257287, completion_time=0.144551561, total_time=0.162808848)
The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=342, total_tokens=417, completion_tokens_details=None, prompt_tokens_details=None, que

**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
The few-shot example should not come from the six letters because the model might copy or memorize information from the actual data instead of learning the format we want.


*2. Why "use null, do not guess" — what did the model do without that instruction?*
“Use null, do not guess” is important because if some information is missing, the model may try to fill it in by itself. Using null makes sure it does not invent information that was not stated.

*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*
Temperature 0 is good for extraction because we want the model to be consistent and accurate, not creative. For creative tasks, a higher temperature can be better because it allows more variety in the answers.
> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [8]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT='''
With the loan application letter and the extracted JSON, you must output:
1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents",
   "flag for senior review") — NOT "approve" or "reject".
final decisions are made by humans so do not make any final decision
'''

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
brief={}

for i in LETTERS:
  extracted= extract_fields(LETTERS.get(i))
  prompt = BRIEF_PROMPT
  prompt = prompt + "\n\nLetter:\n" + LETTERS.get(i)
  prompt = prompt + "\n\nExtracted JSON:\n" + json.dumps(extracted)

  brief[i] = ask_llm(prompt, temperature=0)
print(f'L001: {brief.get("L001")}')

print(f'L002: {brief.get("L002")}')

print(f'L006:{brief.get("L006")}')


The number of tokens the cell called: CompletionUsage(completion_tokens=76, prompt_tokens=362, total_tokens=438, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010062489, prompt_time=0.018240889, completion_time=0.115342263, total_time=0.133583152)
The number of tokens the cell called: CompletionUsage(completion_tokens=355, prompt_tokens=335, total_tokens=690, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.009872747, prompt_time=0.016055829, completion_time=1.028693149, total_time=1.044748978)
The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=322, total_tokens=397, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010300983, prompt_time=0.015573588, completion_time=0.115683526, total_time=0.131257114)
The number of tokens the cell called: CompletionUsage(completion_tokens=280, prompt_tokens=294, total_tokens=574, completion_tokens_details=None, prompt_tokens_details=None, 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*

For L003, the system picked up the strong points such as the applicant having a profitable business, collateral or a guarantor, and a clear repayment plan. For L006, it correctly noticed the risks such as having no collateral, no stated monthly profit, and wanting a large amount of money for different businesses. So I think the system identified the main strengths and red flags well.


*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

The model was not allowed to say “approve” or “reject” because it is only meant to support the loan officer. Practically, the model may not have all the information needed to make the final decision. Ethically, a person should make the final decision because an AI system could make mistakes or unfair decisions that may seriously affect an applicant.



> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [9]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
df.index = ["L001", "L002", "L003", "L004", "L005", "L006"]
letters = ["L001", "L003", "L006"]
table = []
for field in GOLD["L001"]:
    matches = []
    for i in letters:
        model_answer = df.loc[i, field]
        correct_answer = GOLD[i][field]
        if field == "applicant_name":
            match = str(model_answer).lower() == str(correct_answer).lower()
        elif pd.isna(model_answer) and correct_answer is None:
            match = True
        else:
            match = model_answer == correct_answer
        matches.append(match)
    accuracy = sum(matches) / 3
    table.append([field, matches[0], matches[1], matches[2], accuracy])

accuracy_df = pd.DataFrame(
    table,
    columns=["field", "L001", "L003", "L006", "accuracy"]
)

display(accuracy_df)
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

,field,L001,L003,L006,accuracy
0,applicant_name,True,True,True,1.0
1,amount_ghs,True,True,True,1.0
2,purpose,False,False,False,0.0
3,monthly_profit_ghs,True,True,True,1.0
4,has_collateral_or_guarantor,True,True,True,1.0
5,repayment_months,True,True,True,1.0


### Part 4.2 — Reliability: is the system consistent?

In [10]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
def extract_fields(letter_text, temperature=0):
    prompt = f"{EXTRACT_PROMPT}\nletter:{letter_text}"
    answer = ask_llm(prompt, temperature=temperature)
    answer = answer.replace("```json", "")
    answer = answer.replace("```", "")
    answer = answer.strip()
    try:
        result = json.loads(answer)
        return result
    except json.JSONDecodeError:
        print("Parse error")
        return None
temp0 = []
temp1 = []
for i in range(5):
    temp0.append(extract_fields(LETTERS.get("L004"), temperature=0))
for i in range(5):
    temp1.append(extract_fields(LETTERS.get("L004"), temperature=1.0))
valid0 = 0
valid1 = 0
for i in temp0:
    if i is not None:
        valid0 += 1
for i in temp1:
    if i is not None:
        valid1 += 1
unique0 = set()
unique1 = set()
for i in temp0:
    if i is not None:
        unique0.add(json.dumps(i, sort_keys=True))
for i in temp1:
    if i is not None:
        unique1.add(json.dumps(i, sort_keys=True))
print("Temperature 0")
print("Valid JSON:", valid0)
print("Unique results:", len(unique0))
print("Temperature 1.0")
print("Valid JSON:", valid1)
print("Unique results:", len(unique1))
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=342, total_tokens=417, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.009960139, prompt_time=0.016622376, completion_time=0.143100387, total_time=0.159722763)
The number of tokens the cell called: CompletionUsage(completion_tokens=82, prompt_tokens=342, total_tokens=424, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010067098, prompt_time=0.017256351, completion_time=0.15188847, total_time=0.169144821)
The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=342, total_tokens=417, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010113805, prompt_time=0.016711149, completion_time=0.141132487, total_time=0.157843636)
The number of tokens the cell called: CompletionUsage(completion_tokens=75, prompt_tokens=342, total_tokens=417, completion_tokens_details=None, prompt_tokens_details=None, que

### Part 4.3 — Hallucination probing

In [11]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
test1 = "What is the applicant's mother's age?\n\nLetter:\n" + LETTERS.get("L001")
output1 = ask_llm(test1, SUMMARY_PROMPT_V2, temperature=0)
print(f'Test one output: {output1}')

irrelevant_text = "I want to get A in intro to ai "
output2 = extract_fields(irrelevant_text)
print(f'Test 2 output: {output2}')

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
'''Test 1 output:
The loan application is from Akosua Mensah, who is seeking a loan of GHS 8,000 to expand her business. The applicant's mother's age is not mentioned in the application. The applicant has a savings history with the susu scheme and has proposed a repayment plan of GHS 450 per month over 20 months. The applicant's sister, a teacher, has agreed to stand as her guarantor.
PASS — The model correctly admitted that the applicant's mother's age was not provided and did not invent an age.

Test 2 output:
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}
PASS — The extractor returned None for all fields and did not make up any false applicant or any false loan information.'''

The number of tokens the cell called: CompletionUsage(completion_tokens=87, prompt_tokens=211, total_tokens=298, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010206441, prompt_time=0.010187781, completion_time=0.306607964, total_time=0.316795745)
Test one output: The loan application is from Akosua Mensah, who is seeking a loan of GHS 8,000 to expand her business. The applicant's mother's age is not mentioned in the application. The applicant has a savings history with the susu scheme and has proposed a repayment plan of GHS 450 monthly over 20 months. The application includes a guarantor, the applicant's sister, who is a teacher.
The number of tokens the cell called: CompletionUsage(completion_tokens=58, prompt_tokens=244, total_tokens=302, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010243463, prompt_time=0.01158939, completion_time=0.08572159, total_time=0.09731098)
Test 2 output: {'applicant_name': None, 'amount_ghs': None, '

"Test 1 output:\nThe loan application is from Akosua Mensah, who is seeking a loan of GHS 8,000 to expand her business. The applicant's mother's age is not mentioned in the application. The applicant has a savings history with the susu scheme and has proposed a repayment plan of GHS 450 per month over 20 months. The applicant's sister, a teacher, has agreed to stand as her guarantor.\nPASS — The model correctly admitted that the applicant's mother's age was not provided and did not invent an age.\n\nTest 2 output:\n{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}\nPASS — The extractor returned None for all fields and did not make up any false applicant or any false loan information."

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
The model was not allowed to say “approve” or “reject” because it is only meant to support the loan officer. Practically, the model may not have all the information needed to make the final decision. Ethically, a person should make the final decision because an AI system could make mistakes or unfair decisions that may seriously affect an applicant.


*2. What did the reliability experiment show about temperature and production systems?*
The reliability experiment showed that a lower temperature gives more consistent answers, while a higher temperature can cause more variation. For a production system like this, temperature 0 is better because we want the extraction to be stable and predictable.


*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*
My system did not hallucinate in the two tests. When I asked for the applicant’s mother’s age, it said the information was not provided. When I gave irrelevant text to the extractor, it returned None for all the fields instead of making up information. The risk can still be reduced by clearly telling the model to use only information given and return null when something is missing.


> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
Ans: If the bank fully automated the decisions, some applicants could be treated unfairly. For example, someone who does not write very good English but actually has a strong business may look weaker to the system just because of how they wrote the letter.

*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
Ans: Since the letters contain personal information, sending them to a third-party API in another country could create privacy and data protection issues. Before using the system in a real Ghanaian microfinance institution, I would check where the data is stored, who can access it, whether it is kept after processing, and whether this follows Ghana's data protection requirements

*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*
Ans: Two safeguards I would add are human review before any final loan decision and proper logging of the system's outputs so mistakes can be checked later. I would also allow applicants to appeal a decision if they believe the system treated their application unfairly.
> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
Ans: Changing a prompt is similar to changing model hyperparameters because in both cases I am testing different settings to improve the output. The difference is that prompt engineering changes the instructions given to the model, while hyperparameter tuning changes how the model itself learns or behaves

2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
   I would not fully trust this system to run without a human checking the results. The hallucination test influenced me most because even though my system passed the tests, it showed me why the model must be tested for whether it invents missing information. For loan decisions, a human should still make the final decision
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
   Ans: From some of my response.usage results, one request used around 300 tokens. So for 1,000 applications, that would be roughly 300,000 tokens for one call per application. If the system makes several calls for each application, the number would be much higher. This means I would compare API providers based on token prices, reliability and performance before choosing one.
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?
   In Lab 2, I used classical machine learning models such as Linear Regression and Random Forest for prediction tasks. I would use classical ML when I have structured data and the problem can be solved well with simpler models. In Lab 3, I worked with feedforward neural networks using hidden layers, ReLU, and dropout, so I would use a neural network when the problem is more complex and there is enough data to train it. In Lab 4, I used a foundation model API for tasks involving natural language, such as summarizing and extracting information from text. I would not use an API if a simpler model can do the job well, or if privacy, cost, and reliability are major concerns.

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.